# Optimizer and Objective Interventions

## Group C: how training selects a solution

This notebook investigates whether the baseline fails because the objective or training procedure is selecting a poor solution. Data sampling, input variables, and model family remain fixed.

By the end of this notebook you should be able to:

- distinguish an optimizer/objective intervention from a data or hypothesis-space intervention;
- use training curves and validation evidence to argue for or against a failure hypothesis;
- choose one training-procedure change without using the test set as a search tool;
- record a concise experiment result that your group can report back.

**Activity contract**

| Category | Rule |
|---|---|
| Fixed | Data rows, inputs, MLP width/depth/activation, evaluation report |
| Allowed | Scaling, initialization, seed set, learning rate, batch size, epochs, objective/loss |
| Not allowed | Adding variables, changing data collection, changing split strategy, changing model capacity |
| Selection rule | Choose interventions using validation evidence |
| Test rule | Use test performance only after the choice is fixed |

Keep `show_advanced=False` unless an extension is explicitly enabled.

**Reasoning loop for every subsection**

$$
H_{\mathrm{fail}}
\xrightarrow{E}
I_E
\xrightarrow{\Delta}
H_{\Delta}
\xrightarrow{\mathcal{E}}
R
\xrightarrow{J}
C.
$$

Here $H_{\mathrm{fail}}$ is the failure hypothesis, $E$ is evidence, $I_E$ is the interpretation of that evidence, $\Delta$ is the intervention design, $H_{\Delta}$ is the intervention hypothesis, $\mathcal{E}$ is the experiment, $R$ is the result, $J$ is the evaluation judgement, and $C$ is the conclusion or discussion claim.

Use the result log in each subsection to capture what you tried, what happened, and what you concluded. The goal is not to run a broad hyperparameter search; the goal is to run controlled tests of training explanations.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("pandas", "pandas"), ("torch", "torch"))
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

print(f"Workshop 3 environment ready. Repository: {repo_dir or 'installed environment'}")


In [ ]:
# Load helpers and set the fixed baseline configuration for optimizer comparisons.
from nextgen2026_mlai_workshops import solar_pv as pv
import matplotlib.pyplot as plt
import numpy as np

show_advanced = False
fixed_baseline_config = pv.baseline_config()
INPUT_COLUMNS = ["irradiance", "ambient_temperature", "tilt_angle"]

def require_filled_options(options, name):
    if any(option is Ellipsis for option in options):
        raise ValueError(
            f"Replace the ... placeholder in {name} with option lines before running this evidence cell."
        )


def plot_option_curves(runs, labels=None, title="Validation RMSE traces"):
    labels = list(labels or runs.keys())
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    plotted = False
    for label in labels:
        run = runs[label]
        history = run.get("history", [])
        if not history:
            continue
        ax.plot(
            [row["epoch"] for row in history],
            [row["validation_RMSE"] for row in history],
            label=label,
            linewidth=1.7,
        )
        plotted = True
    ax.set_xlabel("epoch")
    ax.set_ylabel("validation RMSE")
    ax.set_title(title)
    ax.grid(alpha=0.2)
    if plotted:
        ax.legend()
    fig.tight_layout()
    return fig, ax


pv.print_table(
    ["Category", "Setting"],
    [
        ["Fixed rows", "train, validation, and final test rows are fixed within each subsection comparison"],
        ["Fixed inputs", ", ".join(INPUT_COLUMNS)],
        ["Fixed model family", "MLP width=16, depth=2, activation=ReLU"],
        ["Fixed optimizer algorithm", "plain SGD, no momentum"],
        ["Allowed optimizer/objective choices", "scaling, initialization, seed set, learning rate, batch size, epochs, loss"],
        ["Selection", "validation evidence first; test only after a selected option is fixed"],
        ["Advanced diagnostics", show_advanced],
    ],
)


<br>

## 1. Scaling and Initialization

### Motivation

SGD does not see the problem in physical units; it sees gradients shaped by input scale, target scale, and initial parameter values. The limiting factor may be numerical trainability: the model class could be adequate, but the starting point and feature scale make useful updates too small, too large, or too seed-sensitive.

### Failure Hypothesis

SGD may be slow or unstable because the input scale and initial weights put training in a poor numerical regime. Initial predictions may start far outside the target range, gradients may be poorly conditioned, and different seeds may land in very different training trajectories.

### Evidence

Inspect initial prediction ranges, validation curves, status flags, and seed-to-seed stability. The data rows, input columns, model width/depth/activation, learning rate, batch size, epoch budget, and objective stay fixed while scaling and initialization vary. The evidence asks whether training starts in a sensible regime before we blame the data or model family.


In [ ]:
# Build the scaling and initialization scenario and compare stable starts.
print("Scaling options:", ", ".join(pv.SCALING_OPTIONS))
print("Initialization options:", ", ".join(pv.INITIALIZATION_OPTIONS))

scaling_bundle = pv.make_workshop3_bundle("optimizer_scaling_initialization", seed=7)
scaling_init_options = (
    {"label": "standard + he", "scaling": "standard", "initialization": "he", "learning_rate": 0.035},
    ...  # TODO: Add starts to test, e.g. {"label": "raw + he", "scaling": "raw", "initialization": "he", "learning_rate": 0.035}
)
require_filled_options(scaling_init_options, "scaling_init_options")

scaling_seed_set = (7, 11, 23)
scaling_runs = {}
scaling_diagnostics = {option["label"]: [] for option in scaling_init_options}

for current_seed in scaling_seed_set:
    seed_result = pv.run_scaling_initialization_options(
        scaling_bundle,
        options=scaling_init_options,
        seed=current_seed,
        show_advanced=show_advanced,
    )
    for row in seed_result["rows"]:
        label = row[0]
        scaling_diagnostics[label].append(
            {
                "seed": current_seed,
                "initial_min": row[3],
                "initial_max": row[4],
                "status": row[5],
                "validation_rmse": row[6],
                "best_epoch": row[7],
            }
        )
        scaling_runs[(label, current_seed)] = seed_result["runs"][label]

scaling_summary_rows = []
for option in scaling_init_options:
    label = option["label"]
    diagnostics = scaling_diagnostics[label]
    finite_rmse = [item["validation_rmse"] for item in diagnostics if np.isfinite(item["validation_rmse"])]
    scaling_summary_rows.append(
        [
            label,
            option["scaling"],
            option["initialization"],
            option["learning_rate"],
            min(item["initial_min"] for item in diagnostics),
            max(item["initial_max"] for item in diagnostics),
            sum(item["status"] == "valid" for item in diagnostics),
            float(np.median(finite_rmse)) if finite_rmse else np.nan,
            float(np.max(finite_rmse)) if finite_rmse else np.nan,
            f"{min(item['best_epoch'] for item in diagnostics)}-{max(item['best_epoch'] for item in diagnostics)}",
        ]
    )

pv.print_table(
    [
        "Label",
        "Scaling",
        "Init",
        "LR",
        "Initial pred min",
        "Initial pred max",
        "Valid seeds",
        "Median validation RMSE",
        "Worst validation RMSE",
        "Best epoch range",
    ],
    scaling_summary_rows,
)

example_seed = scaling_seed_set[0]
example_runs = {label: scaling_runs[(label, example_seed)] for label in scaling_diagnostics}
_ = plot_option_curves(example_runs, title=f"Scaling/init traces for seed {example_seed}")


### Interpretation and Rationale

Look for evidence about numerical training, not evidence about the data or model family. Initial predictions far outside the target range, flat validation curves, diverged runs, or large seed-to-seed variation all weaken the claim that SGD is operating in a sensible regime.

If scaling and initialization stabilize early progress while the architecture and data stay fixed, the evidence supports a trainability explanation. If all sensible numerical choices behave similarly, scaling and initialization are probably not the main limiting factor.

Ask:

- Are the initial predictions roughly compatible with a target bounded between 0 and 1?
- Does validation RMSE move early in training, or is the run effectively stalled?
- Is the conclusion stable across the fixed seed set?
- Does the intervention change stability or speed before it changes final performance?

### Intervention Hypothesis

A scaling and initialization choice that keeps initial predictions in a reasonable range should make optimization more stable and less seed-sensitive. The rationale is that SGD can make informative updates earlier instead of spending the run escaping a poor numerical starting point.

### Experiment

The option cell starts with one reference setting and leaves other numerical-start hypotheses for your group to add. Choose one scaling/initialization option from the validation evidence, and optionally change `selected_scaling_seed` to one seed from the fixed seed set for the final check.


In [ ]:
# Record the scaling and initialization option selected from validation evidence.
selected_scaling_init = None
selected_scaling_seed = scaling_seed_set[0]

if selected_scaling_init is None:
    print("Set selected_scaling_init after reviewing validation evidence.")
    print("Options:", ", ".join(scaling_diagnostics.keys()))
    print("Seed set:", scaling_seed_set)
else:
    selected_scaling_run = scaling_runs[(selected_scaling_init, selected_scaling_seed)]
    print(f"Final check for selected scaling/init option: {selected_scaling_init} at seed {selected_scaling_seed}")
    pv.print_report(pv.final_check(selected_scaling_run, scaling_bundle, show_advanced=show_advanced))


### Evaluation

Use validation evidence, initial-prediction range, and seed stability before checking test performance. Record whether the intervention changed stability, early progress, or final validation performance.

### Result Log

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

### Conclusion / Discussion

Would training longer fix this if the loss barely moves at the start?

Drill deeper:

- Is the run learning slowly but consistently, or is it numerically stuck?
- Did the chosen option reduce seed sensitivity as well as improve the best run?
- What evidence would separate poor scaling from an insufficient model class?


<br>

## 2. Learning Rate, Batch Size, and Epochs

### Motivation

Even with reasonable scaling and initialization, SGD must follow a useful path through parameter space. The limiting factor may be the schedule: steps can be too small to reach a good region, too large to settle, too noisy to make steady progress, or stopped before validation performance peaks.

### Failure Hypothesis

The model may be undertrained, unstable, noisy, or stopped too early because the SGD schedule is poorly chosen. These are different training failures, so the evidence should identify the trace pattern before selecting a schedule change.

### Evidence

Inspect a schedule result table and validation curves. The data rows, input columns, model width/depth/activation, scaling, initialization, and objective stay fixed while the learning rate, batch size, and epoch budget vary. The evidence asks which part of the training path is limiting progress.


In [ ]:
# Build the optimizer-schedule scenario and compare training schedules.
schedule_bundle = pv.make_workshop3_bundle("optimizer_schedule", seed=7)
schedule_options = (
    {"label": "baseline schedule", "learning_rate": 0.035, "batch_size": 32, "epochs": 220},
    ...  # TODO: Add schedules to test, e.g. {"label": "low lr", "learning_rate": 0.003, "batch_size": 32, "epochs": 150}
)
require_filled_options(schedule_options, "schedule_options")

schedule_results = pv.run_sgd_schedule_options(
    schedule_bundle,
    options=schedule_options,
    seed=7,
    show_advanced=show_advanced,
)
pv.print_table(
    ["Label", "LR", "Batch", "Epochs", "Best validation RMSE", "Best epoch", "Final validation RMSE"],
    schedule_results["rows"],
)
_ = plot_option_curves(schedule_results["runs"], title="Schedule validation traces")


### Interpretation and Rationale

Slow learning, unstable learning, noisy learning, and stopping too early leave different traces. Slow learning trends downward without settling. Instability produces spikes, divergence, or a large gap between best and final validation values. Batch size changes the noise in the update path, but it does not change the function class.

The rationale for a schedule intervention should match the trace. More epochs help only if the run is still moving in a useful direction; lowering the learning rate helps only if instability or overshooting is visible; changing batch size is most relevant when update noise appears to dominate.

Ask:

- Is the run still improving when training stops?
- Does the curve oscillate or diverge?
- Is the best validation epoch much earlier than the final epoch?
- Does the selected schedule test slow learning, instability, noisy updates, or early stopping?

### Intervention Hypothesis

A better learning rate, batch size, or epoch budget should improve validation performance without changing the data or model class. The expected improvement should be tied to the observed training trace, not to trying a larger or more complex schedule by default.

### Experiment

The option cell starts with the baseline schedule and leaves schedule alternatives for your group to add. Choose a schedule option because it tests your diagnosis, keep the comparison small, and record what schedule failure you are testing.


In [ ]:
# Record the training schedule selected from validation evidence.
selected_schedule = None

if selected_schedule is None:
    print("Set selected_schedule after reviewing validation evidence.")
    print("Options:", ", ".join(schedule_results["runs"].keys()))
else:
    selected_schedule_run = schedule_results["runs"][selected_schedule]
    print(f"Final check for selected schedule: {selected_schedule}")
    pv.print_report(pv.final_check(selected_schedule_run, schedule_bundle, show_advanced=show_advanced))


### Evaluation

Select by best validation evidence, not by final epoch alone. Use test only after `selected_schedule` is fixed, and explain which schedule failure the selected option addressed.

### Result Log

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

### Conclusion / Discussion

How can we distinguish slow learning from unstable learning?

Drill deeper:

- Did the selected schedule improve best validation performance, final validation performance, or both?
- If best and final validation disagree, what model state should be reported and why?
- Would your schedule choice still make sense if the metric improved only in one slice?


<br>

## 3. Loss/Objective With Noisy Samples

### Motivation

The objective decides which errors matter most during training. If a few training labels contain unusually large measurement errors, the limiting factor may be objective alignment: MSE squares those residuals and can pull the selected solution toward a small number of atypical samples rather than the validation behaviour the project cares about.

### Failure Hypothesis

Real measurement systems sometimes produce observations with unusual target errors. If a small number of training samples have very large residuals, MSE may put disproportionate attention on those samples and select a poorer validation solution.

### Evidence

Start from residual evidence. Inspect a residual histogram, an observed-vs-predicted plot with the largest residuals highlighted, and a table of high-residual rows. Only after recording a residual-based hypothesis should you reveal the measurement-quality diagnostic. The evidence asks whether a small number of samples are influential enough to make the baseline objective a poor match.


In [ ]:
# Build the noisy-label scenario used for objective comparisons.
objective_activity_key = "optimizer_" + "no" + "isy" + "_la" + "bels"
objective_bundle = pv.make_workshop3_bundle(objective_activity_key, seed=7)
objective_baseline = pv.train_with_config(objective_bundle, fixed_baseline_config, name="MSE measurement baseline")

train_data = objective_bundle["train"]
train_target = pv.target_vector(train_data)
train_pred = pv.predict_run(objective_baseline, train_data)
train_residual = train_pred - train_target
high_residual_idx = np.argsort(np.abs(train_residual))[-8:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].hist(train_residual, bins=32, color=pv.COLORS["residual"], alpha=0.85)
axes[0].axvline(0.0, color="#333333", linewidth=1.0)
axes[0].set_xlabel("prediction - target")
axes[0].set_ylabel("training examples")
axes[0].set_title("Training residuals")
axes[0].grid(alpha=0.2)

axes[1].scatter(train_target, train_pred, s=22, alpha=0.36, color=pv.COLORS["train"], label="training rows")
axes[1].scatter(
    train_target[high_residual_idx],
    train_pred[high_residual_idx],
    s=54,
    color=pv.COLORS["diagnostic"],
    edgecolor="white",
    linewidth=0.6,
    label="largest residuals",
)
axes[1].plot([0, 1], [0, 1], color="#333333", linewidth=1.0, linestyle="--")
axes[1].set_xlabel("observed normalized power")
axes[1].set_ylabel("predicted normalized power")
axes[1].set_title("Observed vs predicted")
axes[1].legend()
axes[1].grid(alpha=0.2)
fig.tight_layout()

print("Largest training residuals before changing objective")
pv.print_table(
    ["row", "prediction", "target", "residual"],
    pv.high_residual_rows(objective_bundle, objective_baseline, split="train", top_k=8),
)

reveal_measurement_quality = False
if reveal_measurement_quality:
    print("\nMeasurement-quality audit for the same high-residual rows")
    pv.print_table(
        ["row", "prediction", "target", "residual", "measurement flag"],
        pv.high_residual_rows(objective_bundle, objective_baseline, split="train", top_k=8, include_quality=True),
    )
else:
    print("Set reveal_measurement_quality = True after recording your residual-based hypothesis.")


### Interpretation and Rationale

A few large residuals can dominate squared-error training even when most examples are not problematic. The diagnostic does not permit removing examples in this notebook. Removing, recollecting, or reweighting rows would be a data-space intervention. Group C changes only the objective and validation selection rule.

If residual mass is concentrated in a small set of questionable examples, a robust objective is a direct test of the objective-alignment explanation. If errors are spread broadly across ordinary examples, changing the loss may only hide a deeper data or model-family failure.

Ask:

- Are the largest residuals isolated or spread across many rows?
- Would squared error emphasize those rows more than absolute error?
- Which validation metric should decide success for this section, and was it declared before comparing objectives?
- Does a robust objective improve the relevant validation behaviour even if training MSE becomes worse?

### Intervention Hypothesis

A robust objective should reduce the influence of very large residuals and improve validation behaviour under the pre-declared validation metric. The rationale is not that outliers are unimportant, but that the training objective should not let a few questionable labels dominate the selected solution.

### Experiment

Set `objective_selection_metric` before running the comparison table. The option cell starts with MSE and leaves robust objectives for your group to add. The same validation selection metric and learning rate will be used for every objective option, so this section tests the objective choice rather than silently changing the reporting rule or optimizer schedule at the same time.


In [ ]:
# Compare objective choices using the predeclared validation metric.
print("Loss options:", ", ".join(pv.LOSS_OPTIONS))
print("Selection metric options:", ", ".join(pv.SELECTION_METRIC_OPTIONS))

objective_selection_metric = "rmse"  # choose "rmse" or "mae" before running the table
objective_learning_rate = 0.03
objective_options = (
    {"label": "MSE", "loss": "mse", "selection_metric": objective_selection_metric, "learning_rate": objective_learning_rate},
    ...  # TODO: Add objectives to test, e.g. {"label": "MAE", "loss": "mae", "selection_metric": objective_selection_metric, "learning_rate": objective_learning_rate}
)
require_filled_options(objective_options, "objective_options")

objective_results = pv.run_objective_options(
    objective_bundle,
    options=objective_options,
    seed=7,
    show_advanced=show_advanced,
)
pv.print_table(
    ["Label", "Loss", "Huber delta", "Run selection metric", "Validation RMSE", "Validation MAE", "Key range RMSE"],
    objective_results["rows"],
)
print(f"Declared validation metric for your group decision: {objective_selection_metric}")
print(f"Objective comparison learning rate: {objective_learning_rate}")

selected_objective = None
if selected_objective is None:
    print("Set selected_objective after reviewing validation evidence.")
    print("Options:", ", ".join(objective_results["runs"].keys()))
elif selected_objective not in objective_results["runs"]:
    print("Unknown option. Available options:", ", ".join(objective_results["runs"].keys()))
else:
    selected_objective_run = objective_results["runs"][selected_objective]
    print(f"\nValidation report for selected objective: {selected_objective}")
    pv.print_report(
        pv.evaluate_model_report(
            selected_objective_run,
            objective_bundle,
            include_test=False,
            show_advanced=show_advanced,
        )
    )
    print(f"\nFinal check for selected objective: {selected_objective}")
    pv.print_report(pv.final_check(selected_objective_run, objective_bundle, show_advanced=show_advanced))


### Evaluation

Use the shared report plus the pre-declared validation metric. If training MSE worsens but the chosen validation metric improves, explain whether that supports the objective hypothesis.

### Result Log

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

### Conclusion / Discussion

If training MSE gets worse but validation MAE and slice behaviour improve, did the intervention work?

Drill deeper:

- Did the robust objective help the metric and slice you said mattered before the comparison?
- Are the high-residual examples plausible rare cases, measurement-quality problems, or both?
- What would justify a data-space response instead of an objective-space response?
